In [0]:
# =============================================================================
# Parameters
#   catalog  environment to reset
#   confirm  type the catalog name again to confirm (guards against accidents)
# =============================================================================
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("confirm", "", "Type the catalog name to confirm")

In [0]:
# =============================================================================
# Reset a DEVELOPMENT environment: drop every schema created by the setup notebooks
# -----------------------------------------------------------------------------
# DESTRUCTIVE. Removes tables, volumes (landing files and checkpoints) and the
# audit log. Never part of the Job. After running it, rebuild with:
#   00_setup_* -> 01_source_simulator -> bakehouse_daily job
# The declarative pipeline schema (bakehouse_dlt) is not dropped: its tables are
# managed by the pipeline and are reset with a full refresh instead.
# =============================================================================
import re

PROTECTED_CATALOGS = {"prod"}
SCHEMAS_TO_DROP = ["gold", "silver", "bronze", "ops", "raw"]   # consumers first

catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog!r}")
if catalog in PROTECTED_CATALOGS:
    raise PermissionError(f"Catalog '{catalog}' is protected and can never be reset.")
if dbutils.widgets.get("confirm").strip() != catalog:
    dbutils.notebook.exit(f"Cancelled: type '{catalog}' in 'confirm' to reset this environment.")

for schema in SCHEMAS_TO_DROP:
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")
    print(f"Dropped {catalog}.{schema}")

print("Environment reset. Rebuild: 00_setup_* -> 01_source_simulator -> bakehouse_daily job")